# Coherence Analyses

This notebook contains example of how we can use coherence analyses to identify coherent signals in seismic data. We use data from Brady's Hot Spring geothermal field in Nevada, USA. The data contains both coherent signals (microseismic events) and incoherent noise (background seismic noise). We will compare the performance of different methods for identifying coherent signals, including exact eigenvalue decomposition, SVD approximation, and QR approximation.

In the exact eigenvalue decomposition method, we compute the coherence between pairs of seismic traces using the following formula:

$$Coherence(x,y) = \frac{|P_{xy}|^2}{|P_{xx}||P_{yy}|}$$

where $P_{xy}$ is the cross-power spectral density between signals $x$ and $y$, and $P_{xx}$ and $P_{yy}$ are the power spectral densities of signals $x$ and $y$, respectively.

In the SVD approximation method, we use the formula:

$$Coherence(x,y) = \frac{P_{xy}}{|P_{xx}||P_{yy}|}$$

This representation contains phase information, which can be useful for identifying coherent signals. It is also computationally more efficient than the exact eigenvalue decomposition method. Therefore, in later analyses, we consider the SVD approach as the standard method for comparison against the QR approximation method.


## Data

The data used in this notebook can be downloaded via the [AWS S3 Explorer for the Open Energy Data Initiative](https://data.openei.org/s3_viewer?bucket=nrel-pds-porotomo&prefix=DAS%2FH5%2FDASH%2F). The data is stored in the `nrel-pds-porotomo` bucket and the `DAS/H5/DASH/` prefix. There are separated by dates of recording and arranged in chronological order. The data is stored in the Hierarchical Data Format (HDF) and can be read using the `h5py` package. The file names end in the format `YYMMDDHHmmss.h5` where `YY` is the year, `MM` is the month, `DD` is the day, `HH` is the hour, `mm` is the minute, and `ss` is the second. 

## Import Libraries

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

sys.path.append(os.path.join(os.path.dirname(""), os.pardir, os.pardir))
import coherence_analysis.utils.utils as f

## Parameters for data and tests

In [ ]:
samples_per_sec = 1000
nsensors = 200
start_ch = 3100
nchannels = 2000

win_len = 2.5
overlap = 0

fsize = 15
tick_size = 12
dpi = 150
colors = ["#800000", "#FFD700", "#663399", "#000000", "#008B8B"]

## Load data

We consider two snapshots of data: one coinciding with a **microseismic event** and another with **background noise**.

### Background noise

In [ ]:
file = r"D:\CSM\Mines_Research\Test_data\Brady_Hotspring\PoroTomo_iDAS16043_160312000018.h5"
data, _ = f.load_brady_hdf5(file, normalize="no")

file = r"D:\CSM\Mines_Research\Test_data\Brady_Hotspring\PoroTomo_iDAS16043_160312000048.h5"
data2, _ = f.load_brady_hdf5(file, normalize="no")

data_noise = np.append(data, data2, axis=1)

file = r"D:\CSM\Mines_Research\Test_data\Brady_Hotspring\PoroTomo_iDAS16043_160312000118.h5"
data2, _ = f.load_brady_hdf5(file, normalize="no")

data_noise = np.append(data_noise, data2[:, :10000], axis=1)
data_noise = data_noise[
    start_ch : nchannels + start_ch : int(nchannels / nsensors)
]

### Microseismic event

In [ ]:
file = r"D:\CSM\Mines_Research\Test_data\Brady_Hotspring\PoroTomo_iDAS16043_160314083818.h5"
data, _ = f.load_brady_hdf5(file, normalize="no")

file = r"D:\CSM\Mines_Research\Test_data\Brady_Hotspring\PoroTomo_iDAS16043_160314083848.h5"
data2, _ = f.load_brady_hdf5(file, normalize="no")

data = np.append(data, data2, axis=1)

file = r"D:\CSM\Mines_Research\Test_data\Brady_Hotspring\PoroTomo_iDAS16043_160314083918.h5"
data2, _ = f.load_brady_hdf5(file, normalize="no")

data = np.append(data, data2[:, :10000], axis=1)
data = data[start_ch : nchannels + start_ch : int(nchannels / nsensors)]

### Visualize data snapshots

In [ ]:
v_min = -0.015
v_max = 0.015

plt.figure(figsize=(12, 5), dpi=dpi)
plt.subplot(1, 2, 1)
plt.imshow(
    data_noise,
    cmap="RdBu",
    vmin=v_min,
    vmax=v_max,
    aspect="auto",
    interpolation="none",
    extent=(0, 70, nchannels + start_ch, start_ch),
)

plt.xlabel("Time (seconds)", fontsize=fsize)
plt.ylabel("Channels", fontsize=fsize)
plt.title("Background noise", fontsize=fsize)
# plt.title('Background noise',fontsize=fsize)
plt.xticks(fontsize=tick_size)
plt.yticks(fontsize=tick_size)
# cbar = plt.colorbar()
# cbar.ax.tick_params(labelsize=tick_size)

plt.subplot(1, 2, 2)
plt.imshow(
    data,
    cmap="RdBu",
    vmin=v_min,
    vmax=v_max,
    aspect="auto",
    interpolation="none",
    extent=(0, 70, nchannels + start_ch, start_ch),
)

plt.xlabel("Time (seconds)", fontsize=fsize)
# plt.ylabel("Sensors", fontsize=fsize)
plt.title("Microseismic event", fontsize=fsize)
plt.xticks(fontsize=tick_size)
plt.yticks(fontsize=tick_size)
cbar = plt.colorbar()
cbar.ax.tick_params(labelsize=tick_size)

## Coherence Analysis
### Compute detection parameter

Calculate the detection parameter,

$$\frac{\lambda_i}{\sum_{i=1}^{n}{\lambda_i}}$$

at each frequency. Here $\lambda_i$ is the $i^{th}$ eigenvalue of the coherence matrix.

In [ ]:
## Noise
detection_sig_exact, eigs_exact, freqs = f.coherence(
    data_noise,
    win_len,
    overlap,
    resolution=1,
    sample_interval=1 / samples_per_sec,
    method="exact",
    max_freq=300,
)
detection_sig_svd, eigs_svd, freqs = f.coherence(
    data_noise,
    win_len,
    overlap,
    resolution=1,
    sample_interval=1 / samples_per_sec,
    method="svd",
    max_freq=300,
)
detection_sig_qr, eigs_qr, freqs = f.coherence(
    data_noise,
    win_len,
    overlap,
    resolution=1,
    sample_interval=1 / samples_per_sec,
    method="qr",
    max_freq=300,
)

## Microseismic event
detection_sig_exact2, eigs_exact2, freqs = f.coherence(
    data,
    win_len,
    overlap,
    resolution=1,
    sample_interval=1 / samples_per_sec,
    method="exact",
    max_freq=300,
)
detection_sig_svd2, eigs_svd2, freqs = f.coherence(
    data,
    win_len,
    overlap,
    resolution=1,
    sample_interval=1 / samples_per_sec,
    method="svd",
    max_freq=300,
)
detection_sig_qr2, eigs_qr2, freqs = f.coherence(
    data,
    win_len,
    overlap,
    resolution=1,
    sample_interval=1 / samples_per_sec,
    method="qr",
    max_freq=300,
)

### visualize detection parameter

The goal is to get a general overview within all the frequency range of interest. In the following plot, we look at how the various methods perform in this regard.

In [ ]:
plt.figure(figsize=(20, 5), dpi=dpi)
plt.subplot(1, 3, 1)
plt.plot(
    freqs[1:],
    detection_sig_exact2[1:],
    color="darkviolet",
    label="Microseismic Event",
)
plt.plot(
    freqs[1:],
    detection_sig_exact[1:],
    color="goldenrod",
    alpha=0.9,
    label="Background Noise",
)

# plt.ylabel(r"$\frac{\lambda_1}{\sum_{i=1}^{n}{\lambda_i}}$",fontsize=fsize)
plt.ylabel("Detection parameter", fontsize=fsize)
plt.xlabel("Frequency", fontsize=fsize)
plt.ylim(0, 1)
plt.xticks(fontsize=tick_size)
plt.yticks(fontsize=tick_size)
plt.title("Exact", fontsize=fsize)
plt.legend(fontsize=tick_size)

plt.subplot(1, 3, 2)
plt.plot(
    freqs[1:],
    detection_sig_svd2[1:],
    color="darkviolet",
    label="Microseismic Event",
)
plt.plot(
    freqs[1:],
    detection_sig_svd[1:],
    color="goldenrod",
    alpha=0.9,
    label="Background Noise",
)
# plt.ylabel("Detection parameter", fontsize=fsize)
plt.xlabel("Frequency", fontsize=fsize)
plt.ylim(0, 1)
plt.xticks(fontsize=tick_size)
plt.yticks(fontsize=tick_size)
plt.title("SVD", fontsize=fsize)
plt.legend(fontsize=tick_size)

plt.subplot(1, 3, 3)
plt.plot(
    freqs[1:],
    detection_sig_qr2[1:],
    color="darkviolet",
    label="Microseismic Event",
)
plt.plot(
    freqs[1:],
    detection_sig_qr[1:],
    color="goldenrod",
    alpha=0.9,
    label="Background Noise",
)
# plt.ylabel("Detection parameter", fontsize=fsize)
plt.xlabel("Frequency", fontsize=fsize)
plt.ylim(0, 1)
plt.xticks(fontsize=tick_size)
plt.yticks(fontsize=tick_size)
plt.title("QR", fontsize=fsize)
plt.legend(fontsize=tick_size)

## Decay of eigenvalues corresponding to particular frequency

Beyond the summary over the entire frequency range, we can also look at the decay of eigenvalues corresponding to a particular frequency. Here, we choose a frequency of 14 Hz.

In [ ]:
i = 35
plt.figure(figsize=(20, 5))
plt.subplot(1, 3, 1)
plt.plot(
    np.sort(eigs_exact2[i] / np.sum(eigs_exact2[i]))[::-1],
    "-o",
    color="darkviolet",
    label="Coherent Signal",
)
plt.plot(
    np.sort(eigs_exact[i] / np.sum(eigs_exact[i]))[::-1],
    "-*",
    color="goldenrod",
    label="Background Noise",
)

plt.ylabel("Normalized Eigenvalue", fontsize=fsize)
plt.xlabel("Index, i", fontsize=fsize)
plt.xticks(fontsize=tick_size)
plt.yticks(fontsize=tick_size)
plt.ylim([-0.05, 0.9])
plt.legend(fontsize=tick_size)
plt.title("Exact", fontsize=fsize)

plt.subplot(1, 3, 2)
plt.plot(
    np.sort(eigs_svd2[i] / np.sum(eigs_svd2[i]))[::-1],
    "-o",
    color="darkviolet",
    label="Coherent Signal",
)
plt.plot(
    np.sort(eigs_svd[i] / np.sum(eigs_svd[i]))[::-1],
    "-*",
    color="goldenrod",
    label="Background Noise",
)

plt.ylabel("Normalized Eigenvalue", fontsize=fsize)
plt.xlabel("Index, i", fontsize=fsize)
plt.xticks(fontsize=tick_size)
plt.yticks(fontsize=tick_size)
plt.ylim([-0.05, 0.9])
plt.legend(fontsize=tick_size)
plt.title("SVD approximation", fontsize=fsize)

plt.subplot(1, 3, 3)
plt.plot(
    np.sort(eigs_qr2[i] / np.sum(eigs_qr2[i]))[::-1],
    # np.sort(eigenvals2 / np.sum(eigenvals2))[::-1],
    "-o",
    color="darkviolet",
    label="Coherent Signal",
)
plt.plot(
    np.sort(eigs_qr[i] / np.sum(eigs_qr[i]))[::-1],
    # np.sort(eigenvals / np.sum(eigenvals))[::-1],
    "-*",
    color="goldenrod",
    label="Background Noise",
)

plt.ylabel("Normalized Eigenvalue", fontsize=fsize)
plt.xlabel("Index, i", fontsize=fsize)
plt.xticks(fontsize=tick_size)
plt.yticks(fontsize=tick_size)
plt.ylim([-0.05, 0.9])
plt.legend(fontsize=tick_size)
plt.title("QR approximation", fontsize=fsize)

## Localization of coherent signals

One of the advantages of using the QR decomposition method is that it provides a way to localize coherent signals in the data. We see this by looking at the decay approximated by the QR method compared to the SVD method. In the detection parameter plot below, there are multpile signals that overlap in frequency. Plotting the decay for 8 to 14 Hz show the location of the separate coherent signals.

In [ ]:
plt.figure(figsize=(8, 5), dpi=dpi)

plt.plot(
    freqs[1:],
    detection_sig_svd2[1:],
    "-o",
    markersize=3,
    color="darkviolet",
    label="Standard SVD",
)
plt.plot(
    freqs[1:],
    detection_sig_qr2[1:],
    "-*",
    markersize=6,
    color=colors[0],
    label="QR approximation",
)
# plot vertical line at 14 Hz
plt.axvline(x=14, color=colors[-1], linestyle="--", label="14 Hz")
plt.axvline(x=8.4, color=colors[0], linestyle="--", label="8.4 Hz")
plt.ylabel("Detection parameter", fontsize=fsize)
plt.xlabel("Frequency", fontsize=fsize)
plt.xticks(fontsize=fsize)
plt.yticks(fontsize=fsize)
# plt.title("Proportion of $\lambda_1$ in sum of all eigenvalues",fontsize=fsize)
# plt.title("Microseismic event", fontsize=fsize)
plt.legend(fontsize=fsize)

In [ ]:
i = 35
j = 21
k = 23
l = 25
m = 27
colors = ["#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"]
plt.figure(figsize=(12, 5), dpi=dpi)

plt.subplot(1, 2, 1)
plt.plot(
    eigs_svd2[i] / np.sum(eigs_svd2[i]), "-o", color=colors[4], label="14 Hz"
)
plt.plot(
    eigs_svd2[m] / np.sum(eigs_svd2[m]), "-o", color=colors[3], label="11 Hz"
)
plt.plot(
    eigs_svd2[l] / np.sum(eigs_svd2[l]), "-o", color=colors[2], label="10 Hz"
)
plt.plot(
    eigs_svd2[k] / np.sum(eigs_svd2[k]), "-o", color=colors[1], label="9 Hz"
)
plt.plot(
    eigs_svd2[j] / np.sum(eigs_svd2[j]), "-o", color=colors[0], label="8 Hz"
)
plt.ylabel("Normalized Eigenvalue", fontsize=fsize)
plt.xlabel("Index, i", fontsize=fsize)
plt.xticks(fontsize=tick_size)
plt.yticks(fontsize=tick_size)
plt.ylim([-0.02, 0.9])
plt.legend(fontsize=tick_size)
plt.title("Standard SVD", fontsize=fsize)

plt.subplot(1, 2, 2)
plt.plot(
    eigs_qr2[i] / np.sum(eigs_qr2[i]), "-o", color=colors[4], label="14 Hz"
)
plt.plot(
    eigs_qr2[m] / np.sum(eigs_qr2[m]), "-o", color=colors[3], label="11 Hz"
)
plt.plot(
    eigs_qr2[l] / np.sum(eigs_qr2[l]), "-o", color=colors[2], label="10 Hz"
)
plt.plot(
    eigs_qr2[k] / np.sum(eigs_qr2[k]), "-o", color=colors[1], label="9 Hz"
)
plt.plot(
    eigs_qr2[j] / np.sum(eigs_qr2[j]), "-o", color=colors[0], label="8 Hz"
)
# plt.ylabel("Normalized Eigenvalue", fontsize=fsize)
plt.xlabel("Snapshot in time", fontsize=fsize)
plt.xticks(fontsize=tick_size)
plt.yticks(fontsize=tick_size)
plt.ylim([-0.02, 0.9])
plt.legend(fontsize=tick_size)
plt.title("QR approximation", fontsize=fsize)